# Audion litterointi Hindenburgia varten: whisper-timestamped

Versio 0.92

Luotu 6.3.2024, muokattu 8.10.2025.

Pohjana Whisper for Podcasters v0.2 Oct 2, 2022 by [@UncannyRobot](https://twitter.com/UncannyRobot)



In [ ]:
# @title Alusta litterointiympäristö & Määritä välimuisti
import os
from IPython.display import clear_output

# --- KÄYTTÄJÄN ASETUKSET ---
input_dir = '/content/drive/MyDrive/whisper/input' # Drive-kansio, johon lataat käsiteltävät tiedostot
output_dir = '/content/drive/MyDrive/whisper/output/' # Drive-kansio, johon litteroinnit sisältävä .nhsx-tiedosto tallentuu
# ---------------------------

# Mount Google Drive first to access it
from google.colab import drive
drive.mount('/content/drive')

# Define a persistent cache directory on Google Drive
cache_dir = "/content/drive/MyDrive/whisper/cache"
os.makedirs(cache_dir, exist_ok=True)

# Set environment variables to use the persistent cache for models
os.environ['HF_HOME'] = cache_dir
os.environ['TORCH_HOME'] = cache_dir

# Tarkista ensin, tarvitseeko tehdä mitään
def quick_check_if_any_files_need_transcription(input_dir):
    """Quick check if any audio files need transcription."""
    if not os.path.exists(input_dir):
        return False  # Jos kansiota ei ole, ei tarvitse tehdä mitään
    audio_extensions = ('.wav', '.aiff', '.flac', '.m4a', '.mp4', '.mp3')
    for dirpath, dirnames, filenames in os.walk(input_dir):
        for filename in filenames:
            if filename.lower().endswith(audio_extensions):
                full_path = os.path.join(dirpath, filename)
                if not is_file_already_transcribed_quick(full_path, input_dir):
                    return True
    return False

def is_file_already_transcribed_quick(wav_file_path, input_dir):
    """Quick check without parsing XML fully."""
    filename = os.path.basename(wav_file_path)
    nhsx_filename = filename.replace(os.path.splitext(filename)[1], '.nhsx', 1)
    nhsx_file_path = os.path.join(input_dir, nhsx_filename)
    if os.path.isfile(nhsx_file_path):
        with open(nhsx_file_path, 'r') as f:
            content = f.read()
            if '<Transcription>' in content:
                return True
    return False

if not quick_check_if_any_files_need_transcription(input_dir):
    clear_output()
    print("Kaikki äänitiedostot ovat jo litteroitu. Ei tarvitse asentaa kirjastoja tai ladata malleja.")
    print("Jos haluat käsitellä uusia tiedostoja, lisää ne input-kansioon ja aja tämä solu uudelleen.")
else:
    # Now, install the required packages only if needed
    ! apt install libcublas11
    ! pip install -U whisper-timestamped onnxruntime torchaudio transformers
    clear_output()
    print(f'Valmista on. Välimuisti on asetettu osoitteeseen: {cache_dir}')
    print("Voit nyt jatkaa seuraaviin soluihin.")


In [ ]:
# @title Litteroi
# This is a Python script that reads all specified audio files in the input directory,
# uses Whisper to transcribe them into JSON files,
# then transforms the JSON timestamps into .nhsx XML format
# and inserts them in the .nhsx file

import os
import re
import json
from xml.etree import ElementTree
import whisper_timestamped

def hms_to_seconds(time_str):
    hours, minutes, seconds_milliseconds = time_str.split(':')
    seconds, milliseconds = seconds_milliseconds.split(',')
    return int(hours) * 3600 + int(minutes) * 60 + int(seconds) + float(milliseconds.replace(',', '.')) / 1000

def check_if_any_files_need_transcription(input_dir):
    """Check if any audio files in the input directory need transcription."""
    audio_extensions = ('.wav', '.aiff', '.flac', '.m4a', '.mp4', '.mp3')
    for dirpath, dirnames, filenames in os.walk(input_dir):
        for filename in filenames:
            if filename.lower().endswith(audio_extensions):
                full_path = os.path.join(dirpath, filename)
                if not is_file_already_transcribed(full_path, input_dir):
                    return True
    return False

def generate_srt_files(input_dir, output_dir, model):
    transcripts_dir = os.path.join(output_dir, 'transcripts')
    os.makedirs(transcripts_dir, exist_ok=True)

    audio_extensions = ('.wav', '.aiff', '.flac', '.m4a', '.mp4', '.mp3') # Added new extensions here

    # Recursive walk through all subdirectories
    for dirpath, dirnames, filenames in os.walk(input_dir):
        for filename in filenames:
            if filename.lower().endswith(audio_extensions): # Modified to check against tuple of extensions
                full_path = os.path.join(dirpath, filename)
                # Keep the filename's case. Append .words.json to the original filename.
                output_path = os.path.join(transcripts_dir, filename + '.words.json')


                # Check if transcription already exists in .nhsx file
                if not is_file_already_transcribed(full_path, input_dir):
                    # If not transcribed, check if the JSON file exists
                    if not os.path.isfile(output_path):
                        print(f"Litteroidaan tiedostoa {full_path}")
                        try:
                            # Load audio and transcribe
                            audio = whisper_timestamped.load_audio(full_path)
                            result = whisper_timestamped.transcribe(model, audio, language="fi", vad=True, beam_size=5, best_of=5)
                            # Save to JSON
                            with open(output_path, 'w', encoding='utf-8') as f:
                                json.dump(result, f, ensure_ascii=False, indent=2)
                            print(f"Tallennettu: {output_path}")
                        except Exception as e:
                            print(f"Virhe litteroinnissa: {e}")
                    else:
                        print(f"Ohitetaan tiedosto '{output_path}', koska se on jo olemassa.")
                else:
                    print(f"Ohitetaan tiedosto '{full_path}', koska se on jo litteroitu.")

def process_files(input_dir, output_dir):
    # First, check if any files need transcription
    if not check_if_any_files_need_transcription(input_dir):
        print("Kaikki äänitiedostot ovat jo litteroitu. Ei tarvitse tehdä mitään.")
        return

    # If files need transcription, load the model
    print("Ladataan malli...")
    model = whisper_timestamped.load_model("openai/whisper-large-v3-turbo", device="cuda")
    print("Malli ladattu.")

    # Generate transcriptions
    generate_srt_files(input_dir, output_dir, model)

    # Process .nhsx files
    for filename in os.listdir(input_dir):
        if filename.lower().endswith(".nhsx"):
            with open(os.path.join(input_dir, filename), 'r') as f:
                nhsx_data = f.read()
            xml_elems = ElementTree.fromstring(nhsx_data)
            for session_elem in xml_elems:
                if 'AudioPool' in session_elem.tag:
                    for audio_pool_elem in session_elem:
                        if 'File' in audio_pool_elem.tag:
                            file_elem_name = audio_pool_elem.get('Name')
                            if file_elem_name:
                                # Append .words.json to the audio file name to find the correct transcript file.
                                srt_filename = file_elem_name + '.words.json'
                                transcripts_dir = os.path.join(output_dir, 'transcripts')
                                transcript_files = os.listdir(transcripts_dir)
                                # transcript_files_lower = [file.lower() for file in transcript_files] # Removed lowercasing
                                # if srt_filename in transcript_files_lower: # Changed the check
                                if srt_filename in transcript_files: # Case-sensitive check, or can be replaced with case-insensitive if needed
                                    # index = transcript_files_lower.index(srt_filename) # No longer needed
                                    # actual_srt_filename = transcript_files[index] # No longer needed
                                    actual_srt_filename = srt_filename # Filename should be directly matching now
                                    try:
                                        with open(os.path.join(transcripts_dir, actual_srt_filename), 'r') as f:
                                            srt_data = json.load(f)
                                    except FileNotFoundError:
                                        print(f"Tiedostoa ei löydy: '{srt_filename}'. Ohitetaan…")
                                        continue
                                else:
                                    print(f"No transcript found for '{srt_filename}'. Skipping...")
                                    continue # Skip this loop iteration if srt file wasn't found
                                transcription_elem = audio_pool_elem.find('Transcription')
                                if transcription_elem is not None:
                                    print(f"Tiedosto '{srt_filename}' on jo litteroitu, ei lisätä sitä.")
                                    continue
                                transcription_elem = ElementTree.SubElement(audio_pool_elem, 'Transcription')

                                p_elem = ElementTree.SubElement(transcription_elem, 'p')
                                for segment in srt_data['segments']:
                                    for word in segment['words']:
                                        sentence = word['word'].strip()
                                        start = word['start']
                                        end = word['end']
                                        word_elem = ElementTree.SubElement(p_elem, 'w')
                                        word_elem.set('l', str(end-start))
                                        word_elem.set('s', str(start))
                                        word_elem.set('sp', 'UU')
                                        word_elem.text = sentence

                                nhsx_tree = ElementTree.ElementTree(xml_elems)
                                output_filename = filename.replace('.nhsx', ' litteroitu.nhsx')
                                nhsx_tree.write(os.path.join(output_dir, output_filename), encoding='unicode')
                                print(f"Lisätään tekstitys '{srt_filename}' tiedostoon '{output_filename}'")
                            else:
                                print(f"No 'File' element found in {filename}. Skipping...")

def is_file_already_transcribed(wav_file_path, input_dir): # Function name is a bit misleading now as it handles more than wav
    """
    Checks if an audio file has already been transcribed by examining the corresponding .nhsx file.

    Args:
        wav_file_path (str): Path to the audio file.
        input_dir (str): Path to the directory containing the .nhsx files.

    Returns:
        bool: True if the file has been transcribed, False otherwise.
    """
    filename = os.path.basename(wav_file_path)
    nhsx_filename = filename.replace(os.path.splitext(filename)[1], '.nhsx', 1) # Replace extension with .nhsx
    nhsx_file_path = os.path.join(input_dir, nhsx_filename)

    if os.path.isfile(nhsx_file_path):
        with open(nhsx_file_path, 'r') as f:
            nhsx_data = f.read()
        xml_elems = ElementTree.fromstring(nhsx_data)
        for session_elem in xml_elems:
            if 'AudioPool' in session_elem.tag:
                for audio_pool_elem in session_elem:
                    if 'File' in audio_pool_elem.tag and audio_pool_elem.get('Name') == filename:
                        if audio_pool_elem.find('Transcription') is not None:
                            return True
    return False

process_files(input_dir, output_dir)
clear_output()
print("\nValmista tuli.")
